In [1]:
using LinearAlgebra
using BenchmarkTools
using Random

In [2]:
function Ogita_Aishima(A, U, V)
    m, n = size(A)
    
    # Step 1: Compute residual projectors R, S and intermediate matrix T
    R = I - U * U'       # R ∈ ℝ^{m×m}
    S = I - V * V'       # S ∈ ℝ^{n×n}
    T = U' * A * V       # T ∈ ℝ^{m×n}
    # Extract diagonal elements needed
    r = diag(R)
    s = diag(S)
    t = diag(T)

    # Step 2: Compute approximate singular values σ̃_i
    sigma_tilde = zeros(n)
    for i in 1:n
        denom = 1 - (r[i] + s[i]) / 2
        sigma_tilde[i] = t[i] / denom
    end

    # Step 3: Compute diagonal parts f̃_ii and g̃_ii
    f_tilde = zeros(m, m)
    g_tilde = zeros(n, n)
    for i in 1:n
        f_tilde[i,i] = r[i] / 2
        g_tilde[i,i] = s[i] / 2
    end

    # Step 4: Compute off-diagonal parts for 1 ≤ i,j ≤ n, i ≠ j
    for i in 1:n
        for j in 1:n
            if i != j
                α = T[i,j] + sigma_tilde[j] * R[i,j]
                β = T[j,i] + sigma_tilde[j] * S[i,j]
                denom = sigma_tilde[j]^2 - sigma_tilde[i]^2
                f_tilde[i,j] = (α * sigma_tilde[j] + β * sigma_tilde[i]) / denom
                g_tilde[i,j] = (sigma_tilde[j]^2 * β - sigma_tilde[i]^2 * α) / denom
            end
        end
    end

    # Step 5: Compute F12 for 1 ≤ i ≤ n, n+1 ≤ j ≤ m
    for i in 1:n
        for j in (n+1):m
            f_tilde[i,j] = -T[j,i] / sigma_tilde[i]
        end
    end

    # Step 6: Compute F21 for n+1 ≤ i ≤ m, 1 ≤ j ≤ n
    for i in (n+1):m
        for j in 1:n
            f_tilde[i,j] = R[i,j] - f_tilde[j,i]
        end
    end

    # Step 7: Compute F22 for n+1 ≤ i,j ≤ m
    for i in (n+1):m
        for j in (n+1):m
            f_tilde[i,j] = R[i,j] / 2
        end
    end

    # Step 8: Update U' and V'
    U_prime = U + U * f_tilde
    V_prime = V + V * g_tilde

    # Construct Σ′ with refined singular values on diagonal
    Σ_prime = zeros(m, n)
    for i in 1:n
        Σ_prime[i,i] = sigma_tilde[i]
    end

    return U_prime, Σ_prime, V_prime
end

Ogita_Aishima (generic function with 1 method)

In [12]:
using LinearAlgebra

# Create test matrix A (m × n), with m > n
m, n = 6, 6
A = randn(m, n)

# Step 1: True SVD
U, S, Vt = svd(A)
Σ = Diagonal(S)

# Truncate and corrupt approximate U and V
U_approx = U + 0.05 * randn(m, m)
V_approx = Vt' + 0.05 * randn(n, n)

# Re-orthogonalize the corrupted bases (so they’re still valid subspaces)
U_approx = Matrix(qr(U_approx).Q)
V_approx = Matrix(qr(V_approx).Q)

# Step 2: Run the iterative refinement algorithm
U_refined, Σ_refined, V_refined = Ogita_Aishima(A, U_approx, V_approx)

# Step 3: Compare errors

A_approx_before = U_approx[:,1:n] * Σ[1:n,1:n] * V_approx[:,1:n]'
A_approx_after = U_refined[:,1:n] * Σ_refined[1:n,1:n] * V_refined[:,1:n]'

println("Initial approximation error (Frobenius): ", norm(A - A_approx_before))
println("After refinement error (Frobenius): ", norm(A - A_approx_after))

Initial approximation error (Frobenius): 8.750455346991535
After refinement error (Frobenius): 22.562909228184004
